# Notebook 03 — Baseline Models

**Day 3 deliverable.** Trains Logistic Regression, Random Forest, and XGBoost independently. Reports top-1 accuracy, top-3 accuracy, and macro-F1 for each. Saves all three models.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'

sns.set_theme(style='whitegrid')

In [ ]:
train_df     = pd.read_csv(PROC / 'train_clean.csv')
test_df      = pd.read_csv(PROC / 'test_clean.csv')
feature_cols = joblib.load(PROC / 'feature_cols.pkl')
le           = joblib.load(PROC / 'label_encoder.pkl')

X_train = train_df[feature_cols].values
y_train = le.transform(train_df['prognosis'])
X_test  = test_df[feature_cols].values
y_test  = le.transform(test_df['prognosis'])

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}  |  Classes: {len(le.classes_)}')

## Helper — Top-k Accuracy

In [ ]:
def top_k_accuracy(model, X, y, k=3):
    probs      = model.predict_proba(X)
    top_k_pred = np.argsort(probs, axis=1)[:, -k:]
    return float(np.mean([y[i] in top_k_pred[i] for i in range(len(y))]))


def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    return {
        'Model':     name,
        'Top-1 Acc': round(accuracy_score(y_te, y_pred), 4),
        'Top-3 Acc': round(top_k_accuracy(model, X_te, y_te, k=3), 4),
        'Macro-F1':  round(f1_score(y_te, y_pred, average='macro'), 4),
    }

## Train Baseline Models

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, n_jobs=-1)
xgb = XGBClassifier(
    n_estimators=200, learning_rate=0.1,
    eval_metric='mlogloss', random_state=42, verbosity=0,
)

results = []
for name, model in [('Logistic Regression', lr), ('Random Forest', rf), ('XGBoost', xgb)]:
    print(f'Training {name}...')
    results.append(evaluate(name, model, X_train, y_train, X_test, y_test))

metrics_df = pd.DataFrame(results)
print()
print(metrics_df.to_string(index=False))

## 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [('LR', lr), ('RF', rf), ('XGB', xgb)]:
    scores = cross_val_score(model, X_train, y_train,
                             cv=skf, scoring='f1_macro', n_jobs=-1)
    print(f'{name:4s}  CV macro-F1: {scores.mean():.4f} +/- {scores.std():.4f}')

## Metrics Comparison Chart

In [ ]:
ax = metrics_df.set_index('Model').plot(
    kind='bar', figsize=(10, 5), colormap='Set2', edgecolor='white'
)
ax.set_title('Baseline Model Comparison', fontsize=14)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(PROC / 'plot_baseline_comparison.png', dpi=150)
plt.show()

## Confusion Matrix — Random Forest

In [ ]:
fig, ax = plt.subplots(figsize=(16, 14))
ConfusionMatrixDisplay.from_predictions(
    y_test, rf.predict(X_test),
    display_labels=le.classes_,
    ax=ax, xticks_rotation=90, colorbar=False,
)
ax.set_title('Random Forest — Confusion Matrix (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig(PROC / 'plot_confusion_matrix_rf.png', dpi=150)
plt.show()

## Feature Importance — Random Forest

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).nlargest(25)

fig, ax = plt.subplots(figsize=(10, 7))
importances.sort_values().plot(kind='barh', ax=ax, color='teal', edgecolor='white')
ax.set_title('Top-25 Feature Importances (Random Forest)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(PROC / 'plot_feature_importance.png', dpi=150)
plt.show()

## Save Models

In [ ]:
joblib.dump(lr,  PROC / 'model_lr.pkl')
joblib.dump(rf,  PROC / 'model_rf.pkl')
joblib.dump(xgb, PROC / 'model_xgb.pkl')

print('Saved: model_lr.pkl, model_rf.pkl, model_xgb.pkl')
print('Proceed to 04_ensemble_treatment.ipynb')